## Background Research

**What is HotpotQA?**
HotpotQA is a QA dataset where most questions require reasoning across *two*
Wikipedia paragraphs, not one. Each example includes "supporting facts" —
sentence-level annotations showing exactly which sentences, in which
paragraphs, were needed to answer. This makes it a good testbed for multi-hop
reasoning, and a good stress-test for standard RAG, which is built around
single retrieve-then-generate.

**Why single-shot RAG struggles here**
A standard RAG pipeline embeds the question once, retrieves top-k passages
once, and generates an answer once. If the answer requires a fact from
document A ("Which team did X play for") to know what to search for in
document B ("What year did that team win the championship"), a single
retrieval pass over the *original* question will often miss document B
entirely — the question text never mentions the team name.

**The agentic alternative**
An agent with tool-use can decide, after seeing the first retrieval, that it
needs another search — using an intermediate fact it just learned as the next
query. This turns retrieval from a fixed one-shot step into a controlled loop:
retrieve → reason → decide (answer or retrieve again) → repeat.

**Plan for this project**
1. Build a baseline: single-retrieval RAG on HotpotQA (expected to do
   reasonably on single-hop-friendly questions, poorly on genuine multi-hop
   ones — this is the hypothesis to test, not an assumed result).
2. Build the agentic version: Claude with a `search_documents` tool it can
   call repeatedly, deciding for itself when it has enough evidence.
3. Score both with exact-match and F1 against HotpotQA's gold answers, on the
   same held-out slice, so the comparison is fair.
4. Read through the *actual* transcripts afterward and write up specific
   cases — including at least one clear failure — rather than only reporting
   the aggregate number.

Nothing below this cell is written yet. Numbers get filled in only after code
runs and produces them.

In [2]:
import sys
from pathlib import Path

# Add the repo root (parent of notebooks/) to sys.path so `src` is importable
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

In [3]:
from src.data import load_hotpotqa

data = load_hotpotqa()
print(f"Loaded {len(data)} examples")
example = data[0]
print("Question:", example["question"])
print("Answer:", example["answer"])
print("Num supporting facts:", len(example["supporting_facts"]["sent_id"]))
print("Num context docs:", len(example["context"]["title"]))

c:\Users\edrin\OneDrive\Desktop\Self Initiated Projects\Agentic-rag-research-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 200 examples
Question: What nationality was Oliver Reed's character in the film Royal Flash?
Answer: Prussian
Num supporting facts: 3
Num context docs: 10


## Data Exploration

Loaded a 200-example slice of HotpotQA (distractor config, validation split,
seed=42) via the Hugging Face `datasets` library.

**Note on dataset source:** the original `hotpot_qa` repo used a Python
loading script, which is no longer supported by recent versions of the
`datasets` library (`datasets>=4.0` dropped script-based loading entirely).
The maintainers have since published a Parquet-format version under
`hotpotqa/hotpot_qa`, which this project uses instead — same data, standard
columnar format, no `trust_remote_code` needed.

**First example, to sanity-check the schema:**
- Question: *"What nationality was Oliver Reed's character in the film Royal Flash?"*
- Answer: `Prussian`
- Supporting facts: 3 sentences
- Context documents: 10

This confirms the "distractor" setup in practice: the model is given 10
context documents, but only a subset of sentences within 2 of them are
actually needed (3 supporting facts) — the rest are plausible-looking
distractors. This is exactly the setup that should punish naive single-shot
RAG if it grabs the wrong subset of documents on the first retrieval pass,
and is the property the baseline-vs-agentic comparison in this project is
designed to test.

**Next step:** build the baseline single-retrieval RAG pipeline first, per
the plan above, and score it before touching the agentic version — so the
"does the agent actually help" question has a real number to compare against
rather than an assumed one.

## Baseline: Single-Shot Retrieval RAG

Before building anything agentic, this builds and smoke-tests the baseline
this project needs to beat: standard retrieve-once-generate-once RAG.

**Retrieval:** each of the 10 context documents per question is embedded
with `all-MiniLM-L6-v2` (local, no API cost) and indexed with FAISS
(exact search — the scale here doesn't warrant an approximate index). The
raw question is embedded once and the top-2 most similar documents are
retrieved. Top-2 matches the known structure of HotpotQA distractor
questions (exactly 2 of the 10 documents contain the supporting facts),
giving the baseline its best realistic shot at grabbing the right documents
in a single pass.

**Generation:** the retrieved documents are handed to Gemini 2.5 Flash in
one call, with an explicit instruction to say "Cannot determine from
context" rather than guess if the retrieved documents don't contain the
answer -- this makes retrieval failures visible in the output rather than
papered over by the model inventing a plausible-sounding wrong answer.

**Why build and test this before the agentic version:** if the baseline
already retrieves the right 2 documents most of the time, that would mean
this particular 200-example slice isn't a strong test of multi-hop
retrieval, and the eventual "agent vs baseline" comparison needs to be
read in that light. The single-example test below is a smoke test only --
full 200-example scoring happens next, once this is confirmed to run
correctly.

In [4]:
from src.baseline import answer_baseline

result = answer_baseline(data[0])
print("Question:", result["question"])
print("Gold answer:", result["gold_answer"])
print("Retrieved docs:", result["retrieved_titles"])
print("Docs actually needed:", result["supporting_titles"])
print("Generated answer:", result["generated_answer"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3793.75it/s]
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Question: What nationality was Oliver Reed's character in the film Royal Flash?
Gold answer: Prussian
Retrieved docs: ['Royal Flash (film)', 'Royal Flash']
Docs actually needed: ['Royal Flash (film)', 'Royal Flash (film)', 'Otto von Bismarck']
Generated answer: Cannot determine from context.


**Smoke test result:**

- Question: *"What nationality was Oliver Reed's character in the film Royal
  Flash?"*
- Gold answer: `Prussian`
- Retrieved: `Royal Flash (film)`, `Royal Flash`
- Actually needed: `Royal Flash (film)` (x2 supporting facts), `Otto von Bismarck`
- Generated answer: `"Cannot determine from context."`

This single example already surfaces the exact failure mode this baseline
is expected to have. The retriever correctly grabbed `Royal Flash (film)`
but pulled a similarly-titled but wrong second document (`Royal Flash`,
likely the novel or character entry) instead of `Otto von Bismarck` --
which has no lexical or obvious semantic overlap with the raw question text.
The connection to Bismarck only becomes findable *after* knowing which
historical figure Oliver Reed's character represents, information the
single-shot retriever never has a chance to use, since it only ever
searches once, on the original question.

Worth noting: the model did not hallucinate a plausible-sounding wrong
answer -- it correctly reported it couldn't determine the answer from what
it was given. That's a deliberate prompt design choice (see the baseline
prompt's explicit instruction), and it matters for scoring: this kind of
honest "I don't know" should be easy to distinguish from a confident wrong
guess when we get to the full 200-example evaluation, since it's a
retrieval failure being correctly reported, not a reasoning failure.

This is one example, not a pattern yet -- Commit 5 runs this across all 200
and scores it properly. But it's a strong early signal that this dataset
slice is a legitimate multi-hop stress test, not a soft one.